# 04a: Single-Architecture, More-Seeds Test

B Steele, Colorado State University, ROSSyndicate — last updated Sep 16, 2026

`04_make_models.ipynb`'s production ensemble is 5 seeds x 2 algorithms (XGBoost + LightGBM) x 4 folds = 40 models. A follow-up check on the holdout predictions from that run found the two algorithms agree very closely with each other (0.998 correlation between their 5-seed ensemble predictions) while different seeds disagree noticeably more (~0.99 pairwise correlation) — a variance decomposition attributed ~65% of the ensemble's internal spread to seed and only ~13% to model architecture. That raises a real downstream-application question, worth checking early while the pipeline is still easy to change: **would one algorithm with more seeds capture as much (or more) of the useful variability as two algorithms with fewer seeds, at the same total model count?**

This notebook runs that experiment directly: **XGBoost only, 10 independent seeds** (`601`–`610`, a fresh namespace so as not to collide with `04`'s production seeds `501`–`505`), each with its own complete correlation-pruning + backward-elimination + gap-aware-tuning + training pipeline — the same per-seed pipeline as `04`, minus the joint (cross-algorithm) elimination step, which only makes sense with two algorithms. 10 seeds x 1 algorithm x 4 folds = 40 models, the same total count as the production ensemble, evaluated on the exact same fixed holdout, for a fair apples-to-apples comparison.

XGBoost is the one kept here (not LightGBM) because it was marginally the stronger of the two solo ensembles in `05_evaluate_ensemble.ipynb`'s results (RMSE 1.270 m vs. 1.274 m) — a small, not decisive, margin.

This is an exploratory side-experiment, not a replacement for `04`: its outputs live under `xg_models/04a_single_arch/`, entirely separate from `xg_models/v3_production/`.

## Setup

In [1]:
import json
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("python").resolve()))
import backward_elim as be
import features
import metrics as M
import model_xgb
import spatial_cv
from tuning import gap_aware_tune

be.register_model("xgboost", model_xgb)

AQUAMATCH_DIR = Path("aquamatch_files")
MODELS_DIR = Path("xg_models") / "04a_single_arch"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

TARGET = "harmonized_value"
MODEL_NAME = "xgboost"
MOD = model_xgb
SEEDS = list(range(601, 611))  # 10 seeds, fresh namespace vs. 04's 501-505
N_TRIALS_INITIAL = 25
N_TRIALS_FINAL = 40
ELIM_TOL = 0.005
ELIM_SEED = 47
TUNE_SEED = 47

SITE_COLS = ["elevation_m", "catchment_area_sqkm", "pct_impervious_2006", "pct_urban_2006",
             "pct_forest_2006", "pct_cropland_2006", "pct_wetland_2006", "shore_flag"]
WEATHER_COLS_CURATED = ["precip_mm_prev30", "tmax_degC_prev30"]


def log(seed, msg):
    print(f"[{time.strftime('%H:%M:%S')}][seed{seed}] {msg}", flush=True)

## Load data and build 10 fresh CV-fold arrangements

Reads `03_split_data.ipynb`'s output (the same fixed holdout used everywhere else in this project) and adds 10 new `cvfold_seed{601-610}` columns via `spatial_cv.build_splits` — the holdout itself is rebuilt deterministically from the same `holdout_seed=3`, so it comes back identical to `03`'s; only the CV-pool fold arrangement is new. `04`'s original `cvfold_seed{501-505}` columns are left untouched alongside these.

`features.coarsen_site_features()` reduces `catchment_area_sqkm` (log-scale binned) and the `pct_*_2006` land-cover fractions (rounded to the nearest percentage point) so a heavily-resampled single waterbody's static, per-comid LakeCat values can't act as a de facto waterbody ID for the model to key a memorized SDD value off of. It does not (and isn't meant to) address the separate, already-documented leakage from the handful of waterbodies spanning more than one HUC8 (Lake Powell chief among them) — those stay large and distinct enough to remain identifiable after coarsening by design.

In [ ]:
df = pd.read_parquet(AQUAMATCH_DIR / "base_with_splits.parquet")
df = features.add_spectral_indices(df)
df = features.coarsen_site_features(df)
df = spatial_cv.build_splits(df, ensemble_seeds=SEEDS)
print(f"{len(df):,} rows, {df.shape[1]} columns; holdout n={df['is_holdout'].sum():,}")

## Per-seed pipeline (single algorithm)

Same stages as `04_make_models.ipynb`, minus the cross-algorithm joint elimination step (not applicable with one algorithm) — the single model's own backward-elimination result is the seed's final feature set directly, with no intersection step.

In [3]:
def run_seed_pipeline(seed, df):
    out_dir = MODELS_DIR / f"seed{seed}"
    out_dir.mkdir(parents=True, exist_ok=True)
    t_start = time.time()

    folds = spatial_cv.build_folds(df, seed)
    test = df[df["is_holdout"]].reset_index(drop=True)
    log(seed, f"folds: {[(f.part, len(f.train), len(f.val)) for f in folds]}, holdout n={len(test)}")

    # --- candidate pool + correlation pruning (this seed's own train_val pool) ---
    metadata_cols = {"HUC8", "holdout_part", "is_holdout"} | {f"cvfold_seed{s}" for s in SEEDS}
    extra_cols = set(SITE_COLS) | set(WEATHER_COLS_CURATED) | metadata_cols
    optical_feats = [f for f in features.candidate_feature_list(df) if f not in extra_cols]
    candidates = optical_feats + SITE_COLS + WEATHER_COLS_CURATED

    train_val = pd.concat([f.train for f in folds] + [folds[0].val], ignore_index=True).drop_duplicates(
        subset=["siteSR_id", "date"])
    pruned = features.correlation_prune(train_val, candidates, target=TARGET)
    log(seed, f"candidate pool: {len(candidates)} -> {len(pruned)} after correlation pruning")

    # --- stage 1: initial gap-aware tuning (fixed params for elimination) ---
    log(seed, f"=== initial gap-aware tuning on {len(pruned)}-feature pool ===")
    init_result = gap_aware_tune(MOD, folds, pruned, TARGET, n_trials=N_TRIALS_INITIAL, seed=47)
    fixed_params = init_result["best_params"]
    log(seed, f"val_rmse={init_result['best_score']:.4f} gap={init_result['best_gap']:.4f}")

    # --- stage 2: backward elimination (single algorithm, no joint step) ---
    log(seed, "=== backward elimination ===")
    elim_result = be.backward_eliminate(MODEL_NAME, folds, TARGET, fixed_params, pruned,
                                        weight_fn=None, tol=ELIM_TOL, seed=ELIM_SEED,
                                        log=lambda m: log(seed, m))
    final_feats = elim_result["final_features"]
    log(seed, f"final ({len(final_feats)}): {final_feats}")

    with open(out_dir / "backward_elim_summary.json", "w") as fh:
        json.dump(dict(starting_candidates=candidates, correlation_pruned=pruned,
                       final_features=final_feats, path=elim_result["path"]), fh, indent=2, default=str)

    # --- stage 3: final gap-aware tuning on the final feature set ---
    log(seed, f"=== final gap-aware tuning on {len(final_feats)}-feature set ===")
    tune_result = gap_aware_tune(MOD, folds, final_feats, TARGET, n_trials=N_TRIALS_FINAL, seed=TUNE_SEED)
    log(seed, f"chosen: val_rmse={tune_result['best_score']:.4f} gap={tune_result['best_gap']:.4f}")
    with open(out_dir / f"final_tune_{MODEL_NAME}.json", "w") as fh:
        json.dump({"model": MODEL_NAME, "features": final_feats, **tune_result}, fh, indent=2, default=str)

    # --- stage 4: final unweighted training + evaluation ---
    params = tune_result["best_params"]
    fold_models = MOD.train_fold_models(folds, final_feats, TARGET, params)
    for i, fm in enumerate(fold_models):
        fm.save_model(str(out_dir / f"{MODEL_NAME}_fold{i+1}.json"))

    cv_preds, cv_ys = [], []
    for f, model in zip(folds, fold_models):
        cv_preds.append(MOD.predict_ensemble([model], f.val[final_feats]))
        cv_ys.append(f.val[TARGET].values)
    cv_metrics = M.all_metrics(np.concatenate(cv_ys), np.concatenate(cv_preds))

    test_pred = MOD.predict_ensemble(fold_models, test[final_feats])
    test_metrics = M.all_metrics(test[TARGET].values, test_pred)
    log(seed, f"FINAL: cv_rmse={cv_metrics['rmse']:.4f} holdout_rmse={test_metrics['rmse']:.4f}")

    summary = {MODEL_NAME: dict(features=final_feats, params=params, cv_metrics=cv_metrics, test_metrics=test_metrics)}
    with open(out_dir / "final_eval_summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, default=str)

    r = test[["siteSR_id", "date", "HUC4", "HUC8"]].copy()
    r["seed"] = seed
    r["model"] = MODEL_NAME
    r["y"] = test[TARGET].values
    r["pred"] = test_pred
    r.to_parquet(out_dir / "holdout_predictions.parquet")

    log(seed, f"SEED {seed} DONE in {time.time()-t_start:.0f}s")
    return summary

In [4]:
all_summaries = {}
for seed in SEEDS:
    all_summaries[seed] = run_seed_pipeline(seed, df)

[10:55:58][seed601] folds: [(1, 6803, 2189), (2, 6736, 2256), (3, 6718, 2274), (4, 6719, 2273)], holdout n=2218


[10:55:58][seed601] candidate pool: 45 -> 34 after correlation pruning


[10:55:58][seed601] === initial gap-aware tuning on 34-feature pool ===


[10:58:06][seed601] val_rmse=1.5009 gap=0.5226


[10:58:06][seed601] === backward elimination ===


...

[12:21:39][seed610] final (29): ['atm_corr_LaSRC', 'green_corr7', 'nir_corr7', 'swir1_corr7', 'temp_corr7', 'tmean_degC_prev7', 'srad_Wm2_prev1', 'precip_mm_prev3', 'srad_Wm2_prev7', 'tmin_degC_prev7', 'tmean_degC_prev30', 'tmin_degC_prev30', 'srad_Wm2_prev30', 'BR', 'BG', 'NR', 'fai', 'NDVI', 'NDSSI', 'NDWI', 'MNDWI', 'elevation_m', 'catchment_area_sqkm', 'pct_impervious_2006', 'pct_forest_2006', 'pct_cropland_2006', 'pct_wetland_2006', 'shore_flag', 'tmax_degC_prev30']


[12:21:39][seed610] === final gap-aware tuning on 29-feature set ===


[12:24:52][seed610] chosen: val_rmse=1.4314 gap=0.4555


[12:25:01][seed610] FINAL: cv_rmse=1.4422 holdout_rmse=1.2965


[12:25:01][seed610] SEED 610 DONE in 559s


## Per-seed summary

In [5]:
rows = []
for seed, summary in all_summaries.items():
    s = summary[MODEL_NAME]
    rows.append(dict(seed=seed, n_features=len(s["features"]), cv_rmse=s["cv_metrics"]["rmse"],
                     holdout_rmse=s["test_metrics"]["rmse"], holdout_bias=s["test_metrics"]["bias"]))
pd.DataFrame(rows).sort_values("seed")

,seed,n_features,cv_rmse,holdout_rmse,holdout_bias
0,601,27,1.495615,1.266171,0.417646
1,602,32,1.462250,1.297905,0.438523
2,603,28,1.442651,1.289409,0.434750
3,604,26,1.468134,1.270393,0.402106
4,605,32,1.464870,1.275306,0.425806
5,606,31,1.459375,1.295652,0.448578
6,607,30,1.484161,1.283035,0.442730
7,608,29,1.486745,1.250608,0.331453
8,609,24,1.441668,1.285279,0.444751
9,610,29,1.442246,1.296519,0.448917


## Assemble the 10-seed ensemble and compare against production

40 models here (10 seeds x 1 algorithm x 4 folds) vs. 40 models in `04`'s production ensemble (5 seeds x 2 algorithms x 4 folds) — same total model count, same fixed holdout, so the comparison isolates "more seeds, one algorithm" against "fewer seeds, two algorithms" directly.

In [6]:
rows_04a = []
for seed in SEEDS:
    rows_04a.append(pd.read_parquet(MODELS_DIR / f"seed{seed}" / "holdout_predictions.parquet"))
all_preds_04a = pd.concat(rows_04a, ignore_index=True)

ensemble_04a = all_preds_04a.groupby(["siteSR_id", "date"]).agg(y=("y", "first"), pred=("pred", "mean")).reset_index()
metrics_04a = M.all_metrics(ensemble_04a["y"], ensemble_04a["pred"])
print(f"04a (10 seeds x xgboost only, 40 models): {metrics_04a}")

# production ensemble, for direct comparison (04_make_models.ipynb's saved output)
PROD_DIR = Path("xg_models") / "v3_production"
rows_prod = []
for seed in [501, 502, 503, 504, 505]:
    rows_prod.append(pd.read_parquet(PROD_DIR / f"seed{seed}" / "holdout_predictions.parquet"))
all_preds_prod = pd.concat(rows_prod, ignore_index=True)
ensemble_prod = all_preds_prod.groupby(["siteSR_id", "date"]).agg(y=("y", "first"), pred=("pred", "mean")).reset_index()
metrics_prod = M.all_metrics(ensemble_prod["y"], ensemble_prod["pred"])
print(f"production (5 seeds x xgboost+lightgbm, 40 models): {metrics_prod}")

comparison = pd.DataFrame([
    dict(config="04a: 10 seeds x xgboost", **metrics_04a),
    dict(config="production: 5 seeds x xgboost+lightgbm", **metrics_prod),
])
comparison

04a (10 seeds x xgboost only, 40 models): {'rmse': 1.2668307470242317, 'mae': 0.974251922483962, 'mape': 0.4887013570318846, 'bias': 0.4040844786730668, 'p_bias': 11.403799282861936, 'smape': 0.29727696790565644, 'r2': 0.6150378657540239, 'n': 1851}
production (5 seeds x xgboost+lightgbm, 40 models): {'rmse': 1.2709690265766653, 'mae': 0.9786580894168342, 'mape': 0.4896502759021617, 'bias': 0.3996930215663622, 'p_bias': 11.279866545904042, 'smape': 0.2976288760585134, 'r2': 0.6100048093155789, 'n': 1851}


,config,rmse,mae,mape,bias,p_bias,smape,r2,n
0,04a: 10 seeds x xgboost,1.266831,0.974252,0.488701,0.404084,11.403799,0.297277,0.615038,1851
1,production: 5 seeds x xgboost+lightgbm,1.270969,0.978658,0.489650,0.399693,11.279867,0.297629,0.610005,1851


## Summary

The 10-seed x XGBoost-only ensemble (40 models) reaches holdout RMSE 1.267m / MAE 0.974m / R² 0.615 / bias +0.404m — matching, marginally beating, the production 5-seed x XGBoost+LightGBM ensemble (RMSE 1.271m / MAE 0.979m / R² 0.610 / bias +0.400m) at the same total model count. See `05a_evaluate_single_arch.ipynb` for the full evaluation (HUC8 breakdown, seed-vs-seed spread, SHAP) and the interpretation of what this means for the production pipeline's design going forward.